# Baseline: duplicate purchases before LabStock

Phase 0, steps 3 and 4 of the plan (`docs/plan.md` §5). From the lab's own Quartzy history,
this notebook estimates how often two people bought the same product and what that cost,
before LabStock existed. This is the PI's **"before" number**. It is built to err low, and
it says what it assumed.

It reports two figures:

- **Upper bound:** every request for a product that a *different* person had requested in
  the previous 90 days, at the later request's price. Some of these purchases were needed
  (two projects, stock used up, an urgent experiment), so the upper bound overstates the
  waste.
- **Reviewed figure:** only the pairs that the wet-lab co-owner labels `true_dup` in the
  review spreadsheet (the top ~40 by $). **This is the number to quote.**

What you need:

1. A Quartzy export: `uv run --no-sync python -m scripts.quartzy_probe export` writes
   `data/export-<date>/`. It holds only the lab's own Quartzy labs (`QUARTZY_LAB_IDS`): the
   token also sees any collaborator's lab, and that group's purchases don't belong in this
   figure. The notebook uses the newest export unless the environment variable
   `BASELINE_EXPORT_DIR` names another.
2. Optionally, the Quartzy UI export (Requests > Export Requests, "Submitted" + "All Time")
   as CSV, named in `BASELINE_STATUS_CSV`. It adds the dates when requests were ordered,
   received or cancelled, which the API doesn't provide.

The logic lives in `apps/guard/baseline.py` (pure Python, tested in
`tests/guard/test_baseline.py`). This notebook only calls it and shows tables. Nothing here
writes to Quartzy. Files that contain names go to the git-ignored `data/` folder
(`BASELINE_OUTPUT_DIR` overrides it).

In [ ]:
import os
import re
import sys
from pathlib import Path

# The repository root, whether Jupyter started in notebooks/ or at the top.
ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "apps" / "guard").is_dir())
sys.path.insert(0, str(ROOT))


def from_env(name, default=None):
    """A path from an environment variable, relative to this folder or to the repo."""
    value = os.environ.get(name)
    if not value:
        return default
    path = Path(value).expanduser()
    return path if path.is_absolute() or path.exists() else ROOT / path


def export_order(path):
    return [int(number) for number in re.findall(r"\d+", path.name)]  # export-2026-10-27-2


exports = sorted(
    (p.parent for p in (ROOT / "data").glob("export-*/order_requests.jsonl")), key=export_order
)
EXPORT_DIR = from_env("BASELINE_EXPORT_DIR", exports[-1] if exports else None)
OUTPUT_DIR = from_env("BASELINE_OUTPUT_DIR", ROOT / "data")
STATUS_CSV = from_env("BASELINE_STATUS_CSV")
if EXPORT_DIR is None:
    raise FileNotFoundError(
        "No export: run `python -m scripts.quartzy_probe export` or set BASELINE_EXPORT_DIR"
    )
print("Export:", EXPORT_DIR)
print("Status dates:", STATUS_CSV or "none")

In [ ]:
# Settings: the plan's defaults. Change them here, then re-run the notebook from the top.
WINDOW_DAYS = 90  # two requests at most this many days apart can be duplicates
RECEIVED_DAYS = 14  # "received earlier": received at least this long before the later request
ASSUMED_DELIVERY_DAYS = 14  # used without status dates: the assumed request-to-arrival time
MONEY_UNIT = None  # None = what the export manifest pins; or "minor" (cents), "major", "auto"
LAB_IDS = None  # None = the Quartzy labs in the export manifest; or ids (a list, or "id1,id2")
MIN_DISTINCT_NAMES = 4  # a catalog # used under this many item names may be generic
# Without status dates, an earlier request that is still open today was open when the later
# one was made (only an Admin reverting Received to New breaks this). False = the strict
# reading: those pairs become unknown_open_state.
INFER_OPEN_FROM_CURRENT_STATUS = True

# Column headers of the Quartzy UI export, used only with BASELINE_STATUS_CSV. They aren't
# known until the lab exports one: open the file and fix the right-hand side to match it.
STATUS_COLUMNS = {
    "id": "Link",
    "requested_at": "Date Requested",
    "approved_at": "Date Approved",
    "ordered_at": "Date Ordered",
    "received_at": "Date Received",
    "cancelled_at": "Date Cancelled",
    "lab": "Lab",
    "item_name": "Item Name",
    "catalog_number": "Catalog #",
    "vendor_name": "Vendor",
    "requester": "Requested By",
    "price": "Price",
}

## 1. Load the export

Each Quartzy request becomes one record:

- **Requester:** `requested_by`, or `created_by` when that is empty.
- **Vendor family:** Gibco, Invitrogen and Thermo Fisher are one family; Sigma and
  Millipore are another (see `apps/catalog/data/vendors.yaml`). A vendor the file doesn't
  list gets a family of its own, shown with a leading `?`. Requests that name no vendor
  (blank, N/A, TBD...) share the family `unknown`.
- **Normalized catalog number:** formatting is ignored (`11965-092` and `11965092` are the
  same number) and the pack size is split off where the vendor's numbering allows it.
- **Price:** in dollars (major units).

Quartzy doesn't document whether `total_price` counts cents or dollars. The probe measures
this and pins it in the export's manifest. When no unit is pinned, each request is decided
on its own by comparing the total with unit price x quantity. A total that matches neither
is never guessed: that request falls back to unit price x quantity. The table below counts
where each price came from.

Only requests from the Quartzy labs that the export's manifest lists (`lab_ids`, chosen when
the export was made) are read, or from the ones `LAB_IDS` in the settings names instead (a
list of ids, or one string of them separated by commas, as in `QUARTZY_LAB_IDS`). An older
export that doesn't list its labs is read only if all its requests come from one lab.
Otherwise this cell stops and lists the labs to choose from for `LAB_IDS`.

In [ ]:
from collections import Counter

import pandas as pd
from IPython.display import display

from apps.guard import baseline

pd.set_option("display.max_colwidth", 60)


def pct(share):
    return "n/a" if share is None else f"{share:.0%}"


manifest = baseline.read_manifest(EXPORT_DIR)
records = baseline.load_requests(EXPORT_DIR, money_unit=MONEY_UNIT, lab_ids=LAB_IDS)
labs = Counter(r.lab_name or r.lab_id or "(no lab)" for r in records)
print(
    f"{len(records)} requests; export started {manifest.get('started_at')}, "
    f"complete: {manifest.get('complete')}"
)
print("Requests by Quartzy lab:", ", ".join(f"{lab} {n}" for lab, n in labs.most_common()))
pd.Series(Counter(r.price_basis for r in records), name="requests").rename_axis(
    "price from"
).to_frame()

### Optional: status dates from the Quartzy UI export

The API returns only each request's *current* status, which leaves one question open: when
the second person ordered, was the first request still open? (Quartzy's own warning only
looks at open requests.) The UI export has the dates. Its rows are joined to the API by the
request id when the file has one (a request link is enough). Otherwise they are joined by
lab, item, catalog #, requester, request date and price. Rows that match several requests
are left out rather than guessed.

In [ ]:
status_join = None
if STATUS_CSV:
    records, status_join = baseline.attach_status_dates(records, STATUS_CSV, STATUS_COLUMNS)
    print(status_join)
    joined = f"{status_join.attached} of {status_join.rows} rows joined"
    print(f"{joined}; dates read {status_join.date_order}")
else:
    print("No UI export: whether an earlier request was open comes from its current status.")

## 2. Status paths (Phase 0 step 3)

How far requests get through Quartzy's statuses. In a lab where everyone orders for
themselves, many requests never leave CREATED even after the item arrives. That is why
LabStock infers "probably bought" from a request's age rather than from its status. Stale
open requests (CREATED/APPROVED for more than 30 days, ORDERED/BACKORDERED for more than 60)
are the Lab Admin's clean-up list in Phase 0 step 5. The median times need the UI export.

In [ ]:
paths = baseline.status_paths(records, as_of=manifest.get("started_at"))
display(pd.Series(paths["by_status"], name="requests").rename_axis("current status").to_frame())
print(f"As of {paths['as_of']}; with status dates: {paths['with_status_dates']}")
print(
    "Median days to ordered:",
    paths["median_days_to_ordered"],
    "| to received:",
    paths["median_days_to_received"],
)
print("Stale open:", paths["stale_open"]["rule"])
shares = {
    key: paths[key] for key in ("reached_ordered_or_received", "stuck_in_created", "stale_open")
}
pd.DataFrame(shares).T[["count", "share"]].astype({"count": int, "share": float})

## 3. What can't be checked

A request with no usable catalog number (N/A, TBD, blank, a number Excel rounded away, or
a pasted link) can't be compared, so duplicates among those requests are missed. The same
applies to the generic keys found in step 4, and to a request that names no vendor when its
catalog # is too short (under 7 characters) to compare across vendors. The share of
uncheckable requests appears in the summary. The two tables here are to-do lists for the
normalization rules and for `vendors.yaml`.

In [ ]:
unusable = Counter(r.catalog.raw or "(blank)" for r in records if r.catalog.key is None)
unlisted = Counter(r.vendor_name for r in records if r.vendor_family.startswith("?"))
display(
    pd.Series(unusable, name="requests")
    .sort_values(ascending=False)
    .head(15)
    .rename_axis("catalog # as typed")
    .to_frame()
)
pd.Series(unlisted, name="requests").sort_values(ascending=False).head(15).rename_axis(
    "vendor not in vendors.yaml"
).to_frame()

## 4. Generic catalog keys

Some "catalog numbers" are shared by many different items: custom oligos ordered under one
product code, sequencing services, fees. If they were compared, every primer order would
look like a duplicate of the last one. A key typed under many different item names is a
candidate. Read the table before accepting it: a popular product typed several ways can
also show up. The keys that are kept seed LabStock's generic-key stop-list (GenericCatalogKey).

In [ ]:
candidates = baseline.generic_catalog_candidates(records, min_distinct_names=MIN_DISTINCT_NAMES)
pd.DataFrame(candidates, columns=baseline.GenericCandidate._fields)

In [ ]:
# Keep only the keys that really are generic. After reading the table above, replace this
# line with an explicit list, e.g. GENERIC_KEYS = [("idt", "DNAOLIGO25NM")].
GENERIC_KEYS = [(c.vendor_family, c.catalog_key) for c in candidates]
GENERIC_KEYS

## 5. Duplicate pairs

A **duplicate pair** is two requests for the same product by **different people** within
90 days. Each later request is counted once. A repeat by the same person is listed
separately: it is usually a re-order. Cancelled requests are left out.

"The same product" means one of:

| Tier | Meaning | Example |
|---|---|---|
| T1 | same catalog # within a vendor family | Gibco 11965-092 and Thermo 11965092 |
| T2 | same base #, another pack size | Pen-Strep 15140-122 and 15140148 |
| T1x | same catalog # from another vendor family, or on two requests that name no vendor, when the Quartzy types or item names agree | Gibco 25200-056 and Fisher 25-200-056 |

Each pair gets **one bucket** explaining why Quartzy's own warning didn't stop it. The
warning fires on the same catalog # among *open* requests in the *same* Quartzy Lab. A pair
gets the first bucket that applies:

1. `same_number_same_lab_open`: the warning should have fired. People skip the lookup, or
   order through Quartzy Connect. (`unknown_open_state`: the same, but it can't be told
   whether the earlier request was still open.)
2. `different_quartzy_labs`: the warning only looks inside one Lab.
3. `format_variant`: the same number, typed differently.
4. `pack_variant`: the same product in another pack size.
5. `received_earlier`: the first one had arrived at least 14 days before. The warning
   ignores past purchases.
6. `other_vendor_equivalent`: the same item from another vendor (or from unnamed vendors).
7. `unknown`.

The table doesn't list requesters, but item names can still name people: it is for
checking the analysis, not for sharing.

In [ ]:
result = baseline.find_clusters(
    records,
    window_days=WINDOW_DAYS,
    generic_keys=GENERIC_KEYS,
    received_days=RECEIVED_DAYS,
    assumed_delivery_days=ASSUMED_DELIVERY_DAYS,
    infer_open_from_current_status=INFER_OPEN_FROM_CURRENT_STATUS,
)
print(
    f"{len(result.pairs)} duplicate pairs in {len(result.clusters)} clusters; "
    f"{len(result.repeats)} same-requester repeats"
)
pairs = pd.DataFrame(
    [
        {
            "later requested": f"{p.later.requested_at:%Y-%m-%d}",
            "item": p.later.item_name,
            "vendor": p.later.vendor_name,
            "catalog #": p.later.catalog.raw,
            "earlier catalog #": p.earlier.catalog.raw,
            "gap (days)": p.gap_days,
            "tier": p.tier,
            "bucket": p.bucket,
            "amount": p.amount,
        }
        for p in result.pairs
    ]
)
pairs.sort_values("amount", ascending=False, na_position="last").head(20) if len(pairs) else pairs

## 6. Summary for the PI and the lab (anonymized)

Nobody is named, and duplicates are never counted per person (`docs/privacy.md`). Each
product row says how many different people requested it, with no label that could follow
anyone from one row to the next. The names of everyone in the export are taken out of item,
vendor and type names (they often say who an order is for), along with email addresses and
ids, and dates are given by month. Show the **assumptions** list along with the numbers: it
names the Quartzy labs the figure covers and states what it can't see (for example,
purchases made outside Quartzy).

In [ ]:
summary = baseline.summarize(result, manifest=manifest, anonymize=True)
bound, unchecked = summary["upper_bound"], summary["uncheckable"]
print(
    f"Upper bound: {bound['amount']} {bound['currency'] or ''} over {bound['pairs']} pairs "
    f"({bound['unpriced_pairs']} without a price)"
)
print(
    f"Uncheckable: {pct(unchecked['share'])} of non-cancelled requests, "
    f"{pct(unchecked['amount_share'])} of their spend"
)
pd.DataFrame(summary["by_bucket"]).T.rename_axis("bucket")

In [ ]:
for name in ("by_type", "by_month", "by_gap", "by_tier"):
    display(pd.DataFrame(summary[name]).T.rename_axis(name.removeprefix("by_")))

In [ ]:
pd.DataFrame(summary["top_products"]).set_index("cluster") if summary["top_products"] else None

In [ ]:
repeats = summary["same_requester_repeats"]
print(f"Same-requester repeats (not counted): {repeats['pairs']} pairs, {repeats['amount']}")
display(pd.DataFrame(repeats["top_products"]))
print("Assumptions behind these numbers:")
for line in summary["assumptions"]:
    print("-", line)

In [ ]:
# Optional charts; the tables above carry the same numbers.
try:
    import matplotlib.pyplot as plt
except ImportError:
    plt = None
    print("matplotlib isn't installed, so there are no charts.")

if plt is not None:
    ink, muted, grid, bar = "#0b0b0b", "#52514e", "#e5e4e0", "#2a78d6"
    fig, (by_bucket, by_month) = plt.subplots(1, 2, figsize=(12, 4), layout="constrained")
    for axis, data, title in (
        (by_bucket, summary["by_bucket"], "Upper bound by why Quartzy missed it"),
        (by_month, summary["by_month"], "Upper bound by month of the later request"),
    ):
        names, amounts = list(data), [float(row["amount"]) for row in data.values()]
        if axis is by_bucket:
            bars = axis.barh(names, amounts, color=bar, height=0.6)
            axis.invert_yaxis()
            axis.grid(axis="x", color=grid, linewidth=0.8)
        else:
            bars = axis.bar(names, amounts, color=bar, width=0.6)
            axis.tick_params(axis="x", labelrotation=45)
            axis.grid(axis="y", color=grid, linewidth=0.8)
        axis.bar_label(
            bars, labels=[f"{a:,.0f}" if a else "" for a in amounts], padding=3, color=muted
        )
        axis.set_title(title, loc="left", color=ink)
        axis.spines[["top", "right"]].set_visible(False)
        axis.tick_params(colors=muted)
        axis.set_axisbelow(True)
    plt.show()

## 7. The review spreadsheet for the co-owner

`write_review_csv` writes every pair, largest amount first, into `data/`. The file
**contains names**: it is for the co-owner only and must not be shared. Opening it in Excel
or Google Sheets works. The co-owner fills the `label` column for at least the top ~40
rows:

- `true_dup`: one purchase could have been avoided (shared, borrowed, or a joint order).
- `related_not_dup`: the same product, but both were needed (separate projects, the stock
  was used up, an urgent need).
- `false`: not the same thing (the match is wrong).

`notes` holds a reason whenever the label isn't obvious. Save the result as a CSV next to
the original, with `-labelled` added to the name (or name it in `BASELINE_LABELLED_CSV`).
Re-running this notebook never overwrites a file that already has labels.

In [ ]:
REVIEW_CSV = OUTPUT_DIR / f"baseline-review-{EXPORT_DIR.name}.csv"
LABELLED_CSV = from_env(
    "BASELINE_LABELLED_CSV", REVIEW_CSV.with_name(f"{REVIEW_CSV.stem}-labelled.csv")
)
try:
    baseline.write_review_csv(result.pairs, REVIEW_CSV)
    print(f"Wrote {len(result.pairs)} pairs to {REVIEW_CSV}")
except FileExistsError as error:
    print(f"Kept the existing file: {error}")

## 8. The reviewed figure

Only the pairs labelled `true_dup` count. Unlabelled pairs are **not** extrapolated: if
the co-owner checked 40 of 120 pairs, the reviewed figure covers those 40, and the labelled
share of the upper bound is shown next to it. "Precision" is the share of labelled pairs
that were true duplicates. It is a first estimate of how often the matcher is right.

In [ ]:
if LABELLED_CSV.exists():
    reviews = baseline.read_review_csv(LABELLED_CSV)
    reviewed = baseline.reviewed_summary(result.pairs, reviews, window_days=WINDOW_DAYS)
    figures = {
        name: reviewed[name] for name in ("upper_bound", "labelled", "reviewed", "unlabelled")
    }
    display(pd.DataFrame(figures).T[["pairs", "amount"]])
    print(f"Labelled share of the upper bound: {pct(reviewed['labelled']['share_of_upper_bound'])}")
    print(
        f"Precision of the labelled pairs: {pct(reviewed['precision'])}; "
        f"stale labels: {reviewed['stale_labels']}"
    )
    display(pd.DataFrame(reviewed["by_bucket"]).T.rename_axis("bucket"))
else:
    print(
        "No labelled file yet. Once the co-owner has labelled the review CSV, save it as "
        f"{LABELLED_CSV} and re-run this cell."
    )

## What to show at the lab meeting

- The **reviewed figure**, with the upper bound next to it and the labelled share of the
  upper bound.
- The breakdown by bucket: why Quartzy missed each duplicate. That shows which LabStock
  feature closes which gap.
- The share of requests that can't be checked, and the assumptions list.

Leave out the review spreadsheet, which names people, and the tables in sections 3 to 5:
they show item names and catalog numbers as typed, which can name people too. The plan's
privacy commitments (`docs/privacy.md`) promise lab totals only, with no per-person counts.